In [1]:
# iterate over data folder, read each file with suffix "_sportradar.json" and "_features.csv"
# Read the sportradar data dict from the field "timeline" and convert it into a dataframe
# then find the event_id in the sportradar and features dataframes and insert the column "method" and "zone" from sportradar to features and save it as a the features again

import os
import pandas as pd
import json
import numpy as np
import sys

def read_json(file):
    with open(file, 'r') as f:
        data = json.load(f)
    return data

def read_csv(file):
    return pd.read_csv(file)

def write_csv(df, file):
    df.to_csv(file, index=False)

# def main():
    


In [36]:
data_folder = "../data/**"
import glob
path_sportradar = os.path.join(data_folder, "*_sportradar.json")
path_features = os.path.join(data_folder, "*_features.csv")

# list_of_files_sportradar = glob.glob(, recursive=True)
list_of_files_features = glob.glob(os.path.join(data_folder, "*_features.csv"), recursive=True)
list_of_files_sportradar = glob.glob(os.path.join(data_folder, "*_sportradar.json"), recursive=True)
# glob("../data/processed/**/*.csv", recursive=True)
# both filenames have the same id field in the name "*_id_XXX_*.json" and "*_id_XXX_*.csv"
# so we can iterate over the list of files and find the corresponding file with the same id
map_files = {}
for file in list_of_files_sportradar:
    if "event" in file:
        continue
    id = file.split("_id_")[1].split("_")[0]
    map_files[id] = {"sportradar": file}

for file in list_of_files_features:
    if "event" in file:
        continue
    id = file.split("_id_")[1].split("_")[0]
    map_files[id]["features"] = file

for id, files in map_files.items():
    sportradar = read_json(files["sportradar"])
    if "timeline" not in sportradar:
        print(f"file {files['sportradar']} does not have a timeline field")
        continue

    if "features" not in files:
        print(f"file {files['sportradar']} does not have a corresponding features file")
        continue
    features = read_csv(files["features"])
    timeline = sportradar["timeline"]
    df = pd.DataFrame(timeline)
    df["id"] = df["id"].astype(int)
    features["event_id"] = features["event_id"].astype(int)
    # print("Before:")
    # print(features.columns)
    merged = pd.merge(features, df, left_on="event_id", right_on="id", how="left")
    # print columns names of merged
    # print(merged.columns)
    if "method" in merged.columns and "method" not in features.columns:
        features["method"] = merged["method"].fillna("unknown")
    if "zone" in merged.columns and "zone" not in features.columns:
        features["zone"] = merged["zone"].fillna("unknown")
    # shot type
    if "shot_type" in merged.columns and "shot_type" not in features.columns:
        features["shot_type"] = merged["shot_type"].fillna("unknown")

    print(f"Writing file {files['features']}")

    # print columns names of merged
    # print("After")
    # print(features.columns)
    write_csv(features, files["features"])
    # break


Writing file ../data\processed\gameday_01\kinexon\2023-08-24_gd_01_id_42307421_teams_HCErlangen_vs_TSVHannover-Burgdorf_features.csv
Writing file ../data\processed\gameday_01\kinexon\2023-08-24_gd_01_id_42307423_teams_SGFlensburg-Handewitt_vs_HSVHamburg_features.csv
Writing file ../data\processed\gameday_01\kinexon\2023-08-25_gd_01_id_42307425_teams_HSGWetzlar_vs_SCMagdeburg_features.csv
Writing file ../data\processed\gameday_01\kinexon\2023-08-26_gd_01_id_42307427_teams_ThSVEisenach_vs_BergischerHC_features.csv
Writing file ../data\processed\gameday_01\kinexon\2023-08-27_gd_01_id_42307429_teams_HBWBalingen-Weilstetten_vs_THWKiel_features.csv
Writing file ../data\processed\gameday_01\kinexon\2023-08-27_gd_01_id_42307431_teams_MTMelsungen_vs_FRISCHAUF!Goppingen_features.csv
Writing file ../data\processed\gameday_01\kinexon\2023-08-27_gd_01_id_42307433_teams_VfLGummersbach_vs_TBVLemgoLippe_features.csv
Writing file ../data\processed\gameday_01\kinexon\2023-08-28_gd_01_id_42307435_teams_S